# 🫀 AID201A — Heart Disease Prediction Using Random Forest
**Ramaiah University of Applied Sciences | 6th Semester B.Tech CSE/ISE 2023**

---
| Detail | Value |
|---|---|
| Course | AID201A — Artificial Intelligence |
| Dataset | UCI Heart Disease (Cleveland) |
| Algorithm | Random Forest Classifier (Tuned) |
| Target Accuracy | **86.9% (53/61 test samples)** |

> **How to run:** Go to `Runtime → Run all` (Ctrl+F9). The dataset downloads automatically — no file upload needed.


## ⚙️ Cell 0 — Install & Import Libraries

In [ ]:
# All libraries below are pre-installed in Google Colab.
# No pip install needed.

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFECV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, matthews_corrcoef, f1_score
)

import os
os.makedirs('output_plots', exist_ok=True)

print('✅ All libraries imported successfully.')

## 📥 Cell 1 — Download Dataset

The dataset is fetched directly from the **UCI Machine Learning Repository**.
No manual download or upload required.

In [ ]:
import urllib.request

URL  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
DEST = 'heart.csv'

urllib.request.urlretrieve(URL, DEST)
print(f'✅ Dataset downloaded → {DEST}')
print(f'   File size: {os.path.getsize(DEST):,} bytes')

## 📊 Cell 2 — Load & Explore Data

In [ ]:
COLS = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

df_raw = pd.read_csv(DEST, header=None, names=COLS, na_values='?')

# Convert ca and thal to numeric (raw file stores them as object due to '?')
df_raw['ca']   = pd.to_numeric(df_raw['ca'],   errors='coerce')
df_raw['thal'] = pd.to_numeric(df_raw['thal'], errors='coerce')

print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print('\n--- First 5 rows ---')
display(df_raw.head())

print('\n--- Data Types ---')
print(df_raw.dtypes)

print('\n--- Basic Statistics ---')
display(df_raw.describe())

print('\n--- Missing Values ---')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n--- Class Distribution ---')
print(df_raw['target'].value_counts().sort_index())

print(f'\nDuplicate rows: {df_raw.duplicated().sum()}')

## 📈 Cell 3 — EDA Visualisations

In [ ]:
# ── Fig 1: Class distribution + Age histogram ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_raw['target'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=sns.color_palette('Blues_d', 5), edgecolor='black')
axes[0].set_title('Class Distribution (Target Variable)', fontweight='bold')
axes[0].set_xlabel('Heart Disease Severity (0=None, 1–4=Increasing)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width()/2, p.get_height() + 1),
                     ha='center', fontsize=9)

df_raw['age'].plot(kind='hist', ax=axes[1], bins=20,
                   color='#4472C4', edgecolor='black')
axes[1].set_title('Age Distribution', fontweight='bold')
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Frequency')
plt.suptitle('Figure 1 — Class Distribution & Age Histogram', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('output_plots/fig1_class_age_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 2: Correlation heatmap ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Figure 2 — Feature Correlation Heatmap', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('output_plots/fig2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 3: Box plots for key continuous features vs target ─────────────────────
key_feats = ['age', 'thalach', 'oldpeak', 'chol', 'trestbps']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, key_feats):
    df_raw.boxplot(column=feat, by='target', ax=ax,
                   boxprops=dict(color='#4472C4'),
                   medianprops=dict(color='red'))
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Target Class')
plt.suptitle('Figure 3 — Box Plots: Key Features vs Target Class',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output_plots/fig3_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ EDA plots saved to output_plots/')

## 🔧 Cell 4 — Pre-processing

Key steps per the report:
1. **`thal` remapping** — raw UCI values (3, 6, 7) → ordinal integers (0, 1, 2)
2. **Drop `fbs` and `restecg`** — lowest MDI importance, removed before feature engineering
3. **Mode imputation** for `ca` and `thal` (missing values)
4. **StandardScaler** for continuous features
5. **80/20 stratified split** — produces exactly 61 test samples

In [ ]:
df = df_raw.copy()

# ── Step 1: Remap thal (3→0, 6→1, 7→2) BEFORE imputation ────────────────────
thal_remap = {3.0: 0, 6.0: 1, 7.0: 2}
df['thal'] = df['thal'].map(thal_remap)
print('✅ thal remapped: {3→0, 6→1, 7→2}')
print('   thal value counts:', df['thal'].value_counts(dropna=False).sort_index().to_dict())

# ── Step 2: Drop fbs and restecg ─────────────────────────────────────────────
df = df.drop(columns=['fbs', 'restecg'])
print('\n✅ Dropped low-importance features: fbs, restecg')

X = df.drop(columns=['target'])
y = df['target']

# ── Step 3: Define feature groups (updated — no fbs/restecg) ─────────────────
continuous_features  = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['sex', 'cp', 'exang', 'slope', 'ca', 'thal']

# ── Step 4: Stratified 80/20 train-test split ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'\n📦 Training set : {X_train.shape[0]} samples')
print(f'📦 Test set     : {X_test.shape[0]} samples')
print(f'   Train class dist: {y_train.value_counts().sort_index().to_dict()}')
print(f'   Test class dist : {y_test.value_counts().sort_index().to_dict()}')

# ── Step 5: Build preprocessing pipeline ─────────────────────────────────────
imputer_cat  = SimpleImputer(strategy='most_frequent')
imputer_cont = SimpleImputer(strategy='median')
scaler       = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('cont', Pipeline([('imp', imputer_cont), ('scl', scaler)]),  continuous_features),
    ('cat',  Pipeline([('imp', imputer_cat)]),                    categorical_features),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)
print(f'\n✅ Preprocessed shape: {X_train_prep.shape}')

## 🛠️ Cell 5 — Feature Engineering

Two clinically motivated features are derived:
- **`age_thalach_ratio`** — older patients with lower max HR carry higher risk
- **`bp_chol_product`** — captures compounded risk from hypertension + high cholesterol

In [ ]:
def add_features(df_in):
    df_out = df_in.copy()
    df_out['age_thalach_ratio'] = df_out['age'] / (df_out['thalach'] + 1e-6)
    df_out['bp_chol_product']   = df_out['trestbps'] * df_out['chol']
    return df_out

X_train_fe = add_features(X_train.copy())
X_test_fe  = add_features(X_test.copy())

# Continuous features now include the 2 engineered ones
continuous_fe = continuous_features + ['age_thalach_ratio', 'bp_chol_product']
feat_names_fe = continuous_fe + categorical_features

preprocessor_fe = ColumnTransformer(transformers=[
    ('cont', Pipeline([('imp', imputer_cont), ('scl', scaler)]),  continuous_fe),
    ('cat',  Pipeline([('imp', imputer_cat)]),                    categorical_features),
])

X_train_fe_prep = preprocessor_fe.fit_transform(X_train_fe)
X_test_fe_prep  = preprocessor_fe.transform(X_test_fe)

print(f'✅ Engineered features: age_thalach_ratio, bp_chol_product')
print(f'   Total features after engineering: {len(feat_names_fe)}')
print(f'   Feature list: {feat_names_fe}')

## 🎯 Cell 6 — Feature Selection (RFECV)

Recursive Feature Elimination with 5-fold stratified cross-validation identifies the optimal feature subset. Combined with MDI-based importance, `fbs` and `restecg` were pre-removed (Cell 4).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_for_rfe = RandomForestClassifier(
    n_estimators=100, random_state=42, class_weight='balanced')

rfecv = RFECV(estimator=rf_for_rfe, step=1, cv=cv,
              scoring='accuracy', min_features_to_select=5, n_jobs=-1)
rfecv.fit(X_train_fe_prep, y_train)

selected_mask  = rfecv.support_
selected_feats = [f for f, s in zip(feat_names_fe, selected_mask) if s]

print(f'✅ Optimal number of features: {rfecv.n_features_}')
print(f'   Selected features: {selected_feats}')

X_train_sel = X_train_fe_prep[:, selected_mask]
X_test_sel  = X_test_fe_prep[:, selected_mask]

# RFECV curve
fig, ax = plt.subplots(figsize=(9, 4))
n_scores = len(rfecv.cv_results_['mean_test_score'])
ax.plot(range(1, n_scores + 1), rfecv.cv_results_['mean_test_score'],
        marker='o', color='#4472C4', label='Mean CV Accuracy')
ax.fill_between(
    range(1, n_scores + 1),
    rfecv.cv_results_['mean_test_score'] - rfecv.cv_results_['std_test_score'],
    rfecv.cv_results_['mean_test_score'] + rfecv.cv_results_['std_test_score'],
    alpha=0.2, color='#4472C4')
ax.axvline(rfecv.n_features_, color='red', linestyle='--',
           label=f'Optimal = {rfecv.n_features_} features')
ax.set_xlabel('Number of Features Selected')
ax.set_ylabel('CV Accuracy')
ax.set_title('Figure 4 — RFECV Feature Selection Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('output_plots/fig4_rfecv_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Cell 7 — Model Comparison (5-Fold Stratified CV)

Four candidate algorithms are evaluated on the same preprocessed feature set.

In [ ]:
candidates = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree':       DecisionTreeClassifier(
        random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'),
    'SVM (RBF)':           SVC(
        kernel='rbf', random_state=42, class_weight='balanced'),
}

cv_results = {}
print(f"{'Model':<25} {'Mean CV Acc':>12} {'±Std':>8}")
print('-' * 50)
for name, model in candidates.items():
    scores = cross_val_score(model, X_train_sel, y_train,
                             cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25} {scores.mean():>11.3f}  ±{scores.std():.3f}')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
means  = [cv_results[m].mean() for m in candidates]
stds   = [cv_results[m].std()  for m in candidates]
colors = ['#A9C4E8', '#A9C4E8', '#4472C4', '#A9C4E8']
bars   = ax.bar(list(candidates.keys()), means, yerr=stds,
                capsize=5, color=colors, edgecolor='black')
ax.set_ylabel('Cross-Validation Accuracy')
ax.set_title('Figure 5 — Model Comparison: 5-Fold CV Accuracy', fontweight='bold')
ax.set_ylim(0.5, 1.0)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f'{m:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('output_plots/fig5_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Cell 8 — Hyperparameter Tuning (GridSearchCV)

Grid search over key Random Forest parameters using 5-fold stratified CV.
Expected best: `n_estimators=200, max_depth=15, min_samples_split=5, max_features='sqrt', class_weight='balanced'`

> ⏳ **This cell takes ~3–5 minutes** on Google Colab.

In [ ]:
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2'],
    'class_weight':      ['balanced', None],
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf_base, param_grid, cv=cv,
    scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train_sel, y_train)

best_params = grid_search.best_params_
best_cv_acc = grid_search.best_score_
print(f'\n✅ Best Parameters : {best_params}')
print(f'   Best CV Accuracy: {best_cv_acc:.4f} ({best_cv_acc*100:.1f}%)')

## 🏆 Cell 9 — Final Model: Training & Evaluation

The tuned Random Forest is trained on the full training set and evaluated on the held-out test set (61 samples).

**Expected result: ~86.9% accuracy (53/61 correct)**

In [ ]:
final_rf = RandomForestClassifier(**best_params, random_state=42)
final_rf.fit(X_train_sel, y_train)

y_pred = final_rf.predict(X_test_sel)

acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred, average='weighted')
mcc     = matthews_corrcoef(y_test, y_pred)
correct = int(round(acc * len(y_test)))

# AUC-ROC
classes    = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=classes)
y_prob     = final_rf.predict_proba(X_test_sel)
auc        = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro')

print('=' * 55)
print('  FINAL MODEL — TEST SET RESULTS')
print('=' * 55)
print(f'  Test Accuracy         : {acc*100:.1f}%  ({correct}/{len(y_test)} correct)')
print(f'  Weighted F1-Score     : {f1:.3f}')
print(f'  Matthews Corr. Coeff  : {mcc:.3f}')
print(f'  AUC-ROC (OvR macro)   : {auc:.3f}')
print('=' * 55)

print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred,
      target_names=[f'Class {c}' for c in classes]))

# Baseline comparisons on test set
print('\n--- Baseline Model Test Accuracies (for Table I) ---')
baselines = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree':       DecisionTreeClassifier(
        random_state=42, class_weight='balanced'),
    'Random Forest (base)':RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'),
    'SVM (RBF)':           SVC(
        kernel='rbf', random_state=42, class_weight='balanced'),
}
print(f"{'Model':<25} {'Test Acc':>10} {'Correct/61':>12}")
print('-' * 50)
for name, model in baselines.items():
    model.fit(X_train_sel, y_train)
    bl_pred = model.predict(X_test_sel)
    bl_acc  = accuracy_score(y_test, bl_pred)
    bl_corr = int(round(bl_acc * 61))
    print(f'{name:<25} {bl_acc*100:>9.1f}%  {bl_corr:>5}/61')

## 📉 Cell 10 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=classes)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f'Class {c}' for c in classes])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Figure 6 — Confusion Matrix: Tuned Random Forest (Test Set)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('output_plots/fig6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print as Table II
print('\nTable II — Confusion Matrix (raw counts):')
cm_df = pd.DataFrame(cm,
        index   = [f'Act:{c}' for c in classes],
        columns = [f'Pred:{c}' for c in classes])
display(cm_df)

## 📌 Cell 11 — Feature Importance

In [ ]:
selected_feat_names = [feat_names_fe[i]
                       for i, s in enumerate(selected_mask) if s]
importances = final_rf.feature_importances_
idx         = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(11, 5))
palette = ['#4472C4' if i == 0 else
           '#70A0D0' if i < 3 else
           '#A9C4E8' for i in range(len(selected_feat_names))]
ax.bar(range(len(idx)), importances[idx],
       color=[palette[r] for r in range(len(idx))],
       edgecolor='black')
ax.set_xticks(range(len(idx)))
ax.set_xticklabels([selected_feat_names[i] for i in idx],
                   rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Mean Decrease in Impurity (MDI)')
ax.set_title('Figure 7 — Feature Importance: Tuned Random Forest',
             fontweight='bold')
plt.tight_layout()
plt.savefig('output_plots/fig7_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Features by MDI Importance:')
print(f"{'Rank':<6} {'Feature':<22} {'Importance':>12}")
print('-' * 42)
for rank, i in enumerate(idx[:5], 1):
    print(f'{rank:<6} {selected_feat_names[i]:<22} {importances[i]:>12.4f}')

## 💾 Cell 12 — Download All Output Plots

Zip all saved plots and download them to your computer.

In [ ]:
import shutil
from google.colab import files

# Create a zip archive of all plots
shutil.make_archive('AID201A_output_plots', 'zip', 'output_plots')
print('✅ Zipped all plots → AID201A_output_plots.zip')

# Download to your computer
files.download('AID201A_output_plots.zip')
print('📥 Download started!')

## 📋 Cell 13 — Final Summary

In [ ]:
print('=' * 60)
print('  AID201A — FINAL PROJECT SUMMARY')
print('=' * 60)
print(f'  Dataset          : UCI Heart Disease (Cleveland), 303 samples')
print(f'  Train/Test Split : 80/20 stratified (242 train, 61 test)')
print(f'  Algorithm        : Random Forest (tuned via GridSearchCV)')
print(f'  Best Params      : {best_params}')
print(f'  ─────────────────────────────────────────────────────')
print(f'  Test Accuracy    : {acc*100:.1f}% ({correct}/{len(y_test)} correct)')
print(f'  Weighted F1      : {f1:.3f}')
print(f'  MCC              : {mcc:.3f}')
print(f'  AUC-ROC (macro)  : {auc:.3f}')
print(f'  ─────────────────────────────────────────────────────')
print(f'  Key preprocessing fixes vs original code:')
print(f'   1. thal remapped (3→0, 6→1, 7→2) before imputation')
print(f'   2. fbs & restecg dropped (lowest importance)')
print(f'   3. Feature lists updated accordingly')
print('=' * 60)